# 03. Group-aware split과 dataset audit

목표: multi-camera slice를 개별 sample로 무작위 분리할 때 생기는 leakage를 확인하고, label missingness·frame alignment·privacy를 포함한 평가 설계를 만듭니다.

In [ ]:
from random import Random

episodes = ['E1', 'E2', 'E3', 'E4']
slices = [f'Camera_{index}' for index in range(6)] + ['point_cloud']
samples = [(episode, media_slice) for episode in episodes for media_slice in slices]
rng = Random(20260806)
shuffled = samples.copy(); rng.shuffle(shuffled)
cut = round(len(shuffled) * 0.75)
naive_train, naive_test = shuffled[:cut], shuffled[cut:]
overlap = {episode for episode, _ in naive_train} & {episode for episode, _ in naive_test}
print('naive split episode overlap:', sorted(overlap))
assert overlap  # 같은 scene의 다른 camera가 양쪽에 들어갑니다.

In [ ]:
test_episode = episodes[-1]
group_train = [sample for sample in samples if sample[0] != test_episode]
group_test = [sample for sample in samples if sample[0] == test_episode]
group_overlap = {e for e, _ in group_train} & {e for e, _ in group_test}
print({'train_samples': len(group_train), 'test_samples': len(group_test), 'episode_overlap': group_overlap})
assert not group_overlap and len(group_test) == 7
print('주의: group split은 slice leakage를 막지만 같은 교차로 location leakage는 남습니다.')

In [ ]:
audit = {
    'frame_documents': 560,
    'ground_truth': 270,
    'hd_map': 560,
    'trajectory': 112,
    'declared_classes': 29,
    'observed_classes': 26,
    'detections': 6854,
}
frame_documents = audit['frame_documents']
for field in ('ground_truth', 'hd_map', 'trajectory'):
    print(f'{field:12s} coverage={audit[field] / frame_documents:.1%}')
print('unobserved declared classes:', audit['declared_classes'] - audit['observed_classes'])
assert audit['ground_truth'] < audit['frame_documents']
assert audit['observed_classes'] <= audit['declared_classes']

In [ ]:
source_frames = [28, 26, 30, 28]
encoded_frames = [29, 27, 31, 29]
alignment_issues = []
for index, (source, encoded) in enumerate(zip(source_frames, encoded_frames), 1):
    if source != encoded:
        alignment_issues.append({'episode': index, 'source': source, 'encoded': encoded, 'delta': encoded - source})
print('alignment issues:', alignment_issues)
assert len(alignment_issues) == 4 and all(item['delta'] == 1 for item in alignment_issues)

## 권장 평가표

| 축 | 반드시 보고할 항목 |
|---|---|
| split | episode group, camera slice, location overlap |
| 입력 | 사용 camera, PCD 사용 여부, missing modality |
| label | frame coverage, pseudo-label provenance, class support |
| calibration | reprojection error, perturbation robustness |
| detection | class별 AP와 small-object 성능 |
| map | polyline distance, topology consistency |
| system | latency, memory, sensor-drop failure |
| privacy | face·plate residual review와 redaction 기록 |

Episode가 4개뿐이므로 단일 holdout 숫자보다 leave-one-episode-out 4회 결과를 모두 보고하세요. 하지만 이것도 같은 location을 공유하므로 다른 교차로 test가 최종적으로 필요합니다.